>**⚠️ Kernel:** Click **Select Kernel** (top right) → **Python Environments...** → **base (Python 3.10.20)**



# Metagenomic assembly tutorial

## Verify data files:
The data needed for this sessions is located in the `MGX_assembly` (should be already there):

```bash
/biodata/resources/MGX_assembly
├── raw_reads/
│   ├── 25748_R1.fastq.gz
│   └── 25748_R2.fastq.gz
└── assembly/
    └── contig_demo/
```

<div style="border-left: 4px solid #007acc; padding: 0.5em; background: rgba(0, 122, 204, 0.1); border-radius: 4px;">
<strong>💡 Tip:</strong> Run Bash commands in Jupyter:

- Use `!command` for single bash commands
- Use `%%bash` magic for entire cells of bash code
- All existing `!` commands should work as-is in your devcontainer

</div>

In [ ]:
# Check the two files that will be used for assembly
! ls -ltrha /biodata/resources/MGX_assembly/raw_reads/25748_R1.fastq.gz
! ls -ltrha /biodata/resources/MGX_assembly/raw_reads/25748_R2.fastq.gz

# Optionally check the ready-to-use results
#! ls -ltrha /biodata/resources/MGX_assembly/assembly/contig_demo/

## Verify the environment

Check that `megahit` is available, or install if needed.



In [ ]:
import shutil
import subprocess

if shutil.which("megahit") is None:
    print("Installing megahit...")
    subprocess.run(["micromamba", "install", "-c", "bioconda", "megahit", "-y"], check=True)
else:
    print("megahit is already available")

<div style="border-left: 4px solid #007acc; padding: 0.5em; background: rgba(0, 122, 204, 0.1); border-radius: 4px;">
<strong>📝 Question:</strong> Can this be turned into something re-usable?
</div> <br />

<details>
<summary><strong>🔎 Solution :</strong></summary>

```python
import shutil
import subprocess

def check_and_install(package):
    if shutil.which(package) is None:
        print(f"Installing {package}...")
        subprocess.run(["micromamba", "install", "-c", "bioconda", package, "-y"], check=True)
    else:
        print(f"{package} is already available")

check_and_install("megahit")
```

</details>

A quicker way: use `!` at the beggining of the cell to execute bash commands directly.

In [ ]:
# Check where megahit is installed
! which megahit

# Check the version and usage:
! megahit --help | head -n 13

## Contig assembly with megahit

The following command runs megahit with paired-end reads using params `-1` and `-2`, 8 threads (CPU), and 80% of the available memmory:

In [ ]:
output_dir = "/biodata/resources/MGX_assembly/assembly/contig_demo"
tmp_dir = "/biodata/resources/MGX_assembly/assembly/tmp"

reads_fw = "/biodata/resources/MGX_assembly/raw_reads/25748_R1.fastq.gz"
reads_rv = "/biodata/resources/MGX_assembly/raw_reads/25748_R2.fastq.gz"

!mkdir -p {output_dir}
!mkdir -p {tmp_dir}

# Run megahit assembly
! rm -rf {output_dir}
! megahit --tmp-dir {tmp_dir} -1 {reads_fw} -2 {reads_rv} -t 8 -m 0.8 -o {output_dir} --out-prefix "25748"

## Check the outputs

Output directory provided to megahit:

In [ ]:
# Check the contents of the output directory
! ls /biodata/resources/MGX_assembly/assembly/contig_demo

# Let's store that in a variable for later use
contigs_file = "/biodata/resources/MGX_assembly/assembly/contig_demo/25748.contigs.fa"

First n lines of the `preffix.contigs.fa` file

In [ ]:
# Check the contents of the contigs file
! head -n 6 {contigs_file}

<div style="border-left: 4px solid #007acc; padding: 0.5em; background: rgba(0, 122, 204, 0.1); border-radius: 4px;">
<strong>💡 Tip:</strong> About the fasta format:

- Commonly used to save sequences [DNA/RNA/Protein]
- Each sequence is recorded on two lines:

  1. A header line that starts with `>`, which is the identifier for the sequence
  2. A sequence line: recording the sequence associated with the header above (e.g. nucleotides for DNA reads)

</div>










In [ ]:
# show the first 3 headers (you can get all the headers by removing the -m 3 option)
! grep -m 3 ">" {contigs_file}

<div style="border-left: 4px solid #007acc; padding: 0.5em; background: rgba(0, 122, 204, 0.1); border-radius: 4px;">
<strong>📝 Exercise:</strong> How many contigs were assembled?  
</div>

<br />
<details>
<summary><strong>💡 Hint:</strong></summary>
1. Get the lines with sequence identifiers. <br />
2. Count the lines with word cound (line mode): `wc -l`
</details>

<br />
<details>
<summary><strong>🔎 Solution :</strong></summary>

```bash
grep '<3' | wc -l
```

</details>



In [ ]:
# write your solution here

## Collect the length of each contigs

If we take a look at our previous results:

```bash
>k141_63 flag=0 multi=186.6656 len=1388
>k141_64 flag=0 multi=15.2414 len=489
>k141_16 flag=0 multi=232.4671 len=3485
```

Contigs assembled by megahit have their lengh in the same line as the identifier (e.g `len=1388`). <br /><br />
<ins>**We could:**</ins><br />
1: Iterate over the lines with identifiers to get a string `line` like `k141_63 flag=0 multi=186.6656 len=1388` <br />
2: Split `line` by spaces, then take the last `part` (`len=1388`) <br />
3: Split the `part` by the equal sign `=` and take the second part (`1388`)

In [ ]:
def get_length(fp):
  # a list to save the length of each contig
  length_lst = []
  with open(fp,"r") as f:
    for line in f: # iterate throught each line
      if ">" in line: # process only the header lines (those that start with ">")
        last_part = line.strip().split(" ")[-1]
        length = last_part.split("=")[1]
        length_lst.append(int(length))
  return length_lst

contig_fp = os.path.join(contig_demo_dir,sample_id+".contigs.fa")
length_lst = get_length(contig_fp)
length_lst[:10]
#

## Overview of the length distribution of contigs

Let's use matplotlib to create a histogram of the contig lenghts:

In [ ]:
import matplotlib.pyplot as plt
plt.hist(length_lst, color = 'blue', edgecolor = 'black', bins = 1000)
plt.xscale("log")
plt.show()

<div style="border-left: 4px solid #007acc; padding: 0.5em; background: rgba(0, 122, 204, 0.1); border-radius: 4px;">
<strong>📝 Question :</strong> Do you think this is how most metagenomic assembled contigs look like?
</div>

## Evaluation of the assembly result

Notice that most of the contigs are very short (<1000 bp), which indicates that our assembly is very fragmented

**What is a good sequence assembly?**

Let's start by asking the opposite question: what is a bad assembly?

*   Too fragmented
*   Too long: reads connected directly by head to tail
*   Ideal assembly: A concise set of continuous contigs that explain as much of the input reads as possible.

Commonly used evaluation metrics: N50 and L50


### Calculating the N50 metric


*   Input: a list of contig lengths
*   Output: the N50 values of the assembly
*   Workflow:
> 1. Calculate the total length of the contigs
> 2. Sort the contig lengths from long to short
> 3. Sum up the contig lengths from long to short, until you reach 50% of the total length of all contigs
> 4. Return the length of the last visited contig



In [ ]:
# A function to calculate the total length of the contigs
def calculate_total_length(length_lst):
  total_length = 0
  for L in length_lst:
    total_length = total_length + L
  return total_length

total_length = calculate_total_length(length_lst)
print("The total length is:", total_length)

# Sort a copy of the list from long to short
print("Before sorting:", length_lst[:10])
sorted_lengths = sorted(length_lst, reverse=True)
print("After sorting:", sorted_lengths[:10])

cur_L = sorted_lengths[0]
cur_sum = cur_L
for next_L in sorted_lengths[1:]:
  # when the length sum of visited contigs reach the half of the total length, current L will the our N50
  if cur_sum > 0.5 * total_length:
    break
  cur_sum = cur_sum + next_L
  cur_L = next_L

print("N50 is:", cur_L)

<div style="border-left: 4px solid #007acc; padding: 0.5em; background: rgba(0, 122, 204, 0.1); border-radius: 4px;">
<strong>📝 Exercise:</strong> Let's now calculate the L50 metric.
</div> 

The L50 is defined as the smallest number of contigs whose length sum up to 50% of the total size of all contigs

<details>
<summary><strong>💡 Hint :</strong></summary>

*   Input: a list of contig lengths
*   Output: :50 values of the assembly
*   Workflow:
> 1. Calculate the total length of all contigs together
> 2. Sort the contig lengths from long to short
> 3. Sum up the contig lengths from long to short, until you reach 50% of the total length of all contigs
> 4. Return the total number of visited contigs

</details>

<br />
<details>
<summary><strong>🔎 Solution :</strong></summary>

```python
# A function to calculate the total length of the contigs
def calculate_total_length(length_lst):
  total_length = 0
  for L in length_lst:
    total_length = total_length + L
  return total_length

total_length = calculate_total_length(length_lst)
print("The total length is:", total_length)

# Sort a copy of the list from long to short
print("Before sorting:", length_lst[:10])
sorted_lengths = sorted(length_lst, reverse=True)
print("After sorting:", sorted_lengths[:10])

cur_L = sorted_lengths[0]
cur_idx = 1
cur_sum = cur_L
for next_L in sorted_lengths[1:]:
  # when the length sum of visited contigs reach the half of the total length, current L will the our N50
  if cur_sum > 0.5*total_length:
    break
  cur_sum = cur_sum + next_L
  cur_L = next_L
  cur_idx = cur_idx + 1
print("L50 is:",cur_idx)
```

</details>



### Your solution

In [ ]:
# your solution here

## Discussion:

<div style="border-left: 4px solid #007acc; padding: 0.5em; background: rgba(0, 122, 204, 0.1); border-radius: 4px;">
<strong>📝 Question :</strong> What is a good assembly?
</div>
<br />
<details>
<summary><strong>🔎 Solution :</strong></summary>

 <ul>
  <li>N50: the larger the better</li>
  <li>L50: the smaller the better</b></li>
</ul> 

</details>

### Other tools to check/verify the quality of our assembled genomes/contigs

1: MetaQuast: https://quast.sourceforge.net/index.html

2: CheckM: https://ecogenomics.github.io/CheckM/

### Other useful tools

1: seqkit (stats, search, grep/search, translate, etc): https://github.com/shenwei356/seqkit

2: CoverM (coverage, Count/RPKM/TPM): https://github.com/wwood/CoverM